# 00 — Camera Calibration Pipeline: Setup & Smoke Test

**Project:** Camera Calibration and Image Dewarping  
**Author:** Sherif Abd El-Rady — Computer Vision Engineer

---

## The Problem

Every physical camera introduces two fundamentally different types of geometric error that distort the relationship between the real world and the captured image:

| Error Type | Cause | Mathematical Model | Tool |
|---|---|---|---|
| **Lens distortion** | Imperfect optics bending light non-linearly | Brown-Conrady radial + tangential polynomial | Intrinsic calibration (K, D) |
| **Perspective distortion** | Camera viewing angle | Planar projective transform | Homography (H) |

These are **mathematically and physically independent** — applying one correction has zero effect on the other. This project implements both, with a clear separation between them.

## The Pipeline

```
Checkerboard images
      │
      ▼
Corner detection  (Zhang's method — cv2.findChessboardCorners + sub-pixel refinement)
      │
      ▼
Intrinsic estimation  →  K (3×3) + D (1×5)
      │
      ▼
Calibration profile  →  YAML (portable, camera-specific, reusable)
      │
      ├── Lens undistortion    (cv2.remap with precomputed maps)
      │
      └── Perspective rectification  (four-point homography)
```

## Reference

- **Zhang (2000)** — *A Flexible New Technique for Camera Calibration*, IEEE TPAMI.  
  https://doi.org/10.1109/34.888718  
- OpenCV Camera Calibration: https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html


In [ ]:
import sys
from pathlib import Path

# ── Package versions ──────────────────────────────────────────────────────────
import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import yaml

print(f'Python  : {sys.version.split()[0]}')
print(f'OpenCV  : {cv2.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'Matplotlib: {matplotlib.__version__}')

In [ ]:
# ── calibvision package import ────────────────────────────────────────────────
SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

import calibvision
print(f'calibvision {calibvision.__version__}')
print('API:', calibvision.__all__)

In [ ]:
# ── Smoke test: synthetic checkerboard corner detection ───────────────────────
# Validates OpenCV and the calibvision package without any real camera images.

def make_checkerboard(cols: int, rows: int, square_px: int = 60, border: int = 40) -> np.ndarray:
    h = rows * square_px + 2 * border
    w = cols * square_px + 2 * border
    board = np.full((h, w), 255, dtype=np.uint8)
    for r in range(rows + 1):
        for c in range(cols + 1):
            if (r + c) % 2 == 0:
                y0, x0 = border + r * square_px, border + c * square_px
                board[y0:y0+square_px, x0:x0+square_px] = 0
    return board

COLS, ROWS = 9, 6
board = make_checkerboard(COLS, ROWS)

found, corners = cv2.findChessboardCorners(board, (COLS, ROWS))
assert found, 'Smoke test FAILED — corner detection broken'
assert len(corners) == COLS * ROWS

vis = cv2.cvtColor(board, cv2.COLOR_GRAY2BGR)
cv2.drawChessboardCorners(vis, (COLS, ROWS), corners, found)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].imshow(board, cmap='gray'); axes[0].set_title('Synthetic checkerboard'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Detected {COLS}×{ROWS} = {COLS*ROWS} inner corners'); axes[1].axis('off')
plt.suptitle('✓ Smoke test passed', fontsize=12, color='green')
plt.tight_layout()
plt.show()

print('✓ All checks passed — proceed to Notebook 01')